In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

# Leer Silver
silver = spark.table(
    "world_bank.costa_rica.silver_economic_indicators"
)

# Leer Gold
gold = spark.table(
    "world_bank.costa_rica.gold_economic_summary"
)

print("Tablas cargadas correctamente")

#Crear una lista para almacenar resultados
resultados = []

def registrar_prueba(nombre, errores):
    estado = "PASS" if errores == 0 else "FAIL"

    resultados.append((
        nombre,
        estado,
        int(errores)
    ))

    print(f"{nombre}: {estado} | Errores: {errores}")

In [0]:
# Comprobaremos que no existan registros repetidos para el mismo país, año e indicador.
duplicados = (
    silver
    .groupBy("country_code", "year", "indicator_code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

registrar_prueba("Duplicados Silver", duplicados)

In [0]:
# Prueba de valores nulos
nulos = silver.filter(
    F.col("value").isNull()
).count()

registrar_prueba("Valores nulos Silver", nulos)

In [0]:
# Validar rango de años
anios_invalidos = silver.filter(
    (F.col("year") < 2000) |
    (F.col("year") > 2025) |
    F.col("year").isNull()
).count()

registrar_prueba("Rango de años", anios_invalidos)

In [0]:
# Comprobaremos que existan exactamente los cuatro indicadores que seleccionamos.
indicadores_esperados = [
    "NY.GDP.MKTP.KD.ZG",
    "FP.CPI.TOTL.ZG",
    "SL.UEM.TOTL.ZS",
    "SP.POP.TOTL"
]

indicadores_reales = [
    fila["indicator_code"]
    for fila in silver.select("indicator_code").distinct().collect()
]

faltantes = len(
    set(indicadores_esperados) - set(indicadores_reales)
)

inesperados = len(
    set(indicadores_reales) - set(indicadores_esperados)
)

registrar_prueba(
    "Indicadores económicos",
    faltantes + inesperados
)

In [0]:
# Validar población
poblacion_invalida = silver.filter(
    (F.col("indicator_code") == "SP.POP.TOTL") &
    (F.col("value") <= 0)
).count()

registrar_prueba("Población positiva", poblacion_invalida)

In [0]:
# La tabla gold_economic_summary debe contener una fila por cada año presente en Silver.
anios_silver = silver.select("year").distinct().count()

anios_gold = gold.select("year").distinct().count()

diferencia = abs(anios_silver - anios_gold)

registrar_prueba(
    "Consistencia Silver-Gold",
    diferencia
)

In [0]:
# Crear DataFrame de resultados
df_quality = spark.createDataFrame(
    resultados,
    ["test_name", "status", "error_count"]
)

# Agregar fecha de ejecución
df_quality = df_quality.withColumn(
    "execution_timestamp",
    F.current_timestamp()
)

display(df_quality)

In [0]:
# Guardar en Delta Lake
df_quality.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "world_bank.costa_rica.data_quality_results"
    )

print("Resultados de calidad guardados")